# 가설 검증: 구매가 중단된 반복구매 카테고리 쿠폰

> 고객이 반복 구매하던 카테고리의 구매가 중단됐을 때 해당 카테고리 쿠폰을 제공하면, 쿠폰을 받지 않은 유사 고객보다 8주 이내 해당 카테고리 재구매율이 높을 것이다.

## 고정 기준
- 과거 반복구매: 캠페인 시작 182~29일 전 서로 다른 장바구니에서 3회 이상 구매
- 구매 중단: 시작 직전 28일 동안 해당 카테고리 구매 0회
- 처리: 캠페인 배정 + 캠페인 쿠폰에 해당 중단 카테고리 포함
- 비교: 같은 카테고리가 중단됐고 과거 구매 특성이 유사하지만 분석 전후 캠페인 미수신
- 1차 결과: 시작 후 56일 이내 해당 카테고리 재구매 여부
- 2차 결과: 해당 카테고리 56일 매출, 전체 8주 활동 주차 수, 전체 56일 매출
- 쿠폰 카테고리가 50개를 초과하는 광범위 캠페인은 카테고리 타기팅으로 보기 어려워 제외

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",70)
PROJECT_DIR=Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd()
OUTPUT_DIR=PROJECT_DIR/"data"/"processed"
HISTORY_DAYS=182
RECENT_DAYS=28
FOLLOWUP_DAYS=56
MIN_HISTORY_BASKETS=3
MAX_CAMPAIGN_CATEGORIES=50
MIN_TREATED_PAIRS=10
DATASET_END_DAY=711
RANDOM_SEED=42
MATCH_CALIPER_SQUARED=1.0

## 1. 데이터 연결과 캠페인 품질 필터

In [ ]:
tx=pd.read_csv(PROJECT_DIR/"transaction_data.csv",usecols=["household_key","BASKET_ID","DAY","WEEK_NO","PRODUCT_ID","SALES_VALUE"],
    dtype={"household_key":"int32","BASKET_ID":"int64","DAY":"int16","WEEK_NO":"int16","PRODUCT_ID":"int32","SALES_VALUE":"float32"})
products=pd.read_csv(PROJECT_DIR/"product.csv",usecols=["PRODUCT_ID","COMMODITY_DESC"])
tx=tx.merge(products,on="PRODUCT_ID",how="left",validate="many_to_one")
campaign_desc=pd.read_csv(PROJECT_DIR/"campaign_desc.csv")
campaign_table=pd.read_csv(PROJECT_DIR/"campaign_table.csv")
coupons=pd.read_csv(PROJECT_DIR/"coupon.csv").drop_duplicates()
coupon_categories=coupons.merge(products,on="PRODUCT_ID",how="left").dropna(subset=["COMMODITY_DESC"])
category_count=coupon_categories.groupby("CAMPAIGN")["COMMODITY_DESC"].nunique().rename("coupon_categories")
valid_campaigns=campaign_desc.loc[campaign_desc["START_DAY"]+FOLLOWUP_DAYS-1<=DATASET_END_DAY].merge(category_count,on="CAMPAIGN",how="left")
valid_campaigns=valid_campaigns.loc[valid_campaigns["coupon_categories"].between(1,MAX_CAMPAIGN_CATEGORIES)].copy()
print(f"분석 가능 캠페인: {len(valid_campaigns)}개")

## 2. 캠페인별 구매 중단 카테고리와 8주 성과 생성

In [ ]:
def lapsed_category_cohort(start_day):
    history_start=max(1,start_day-HISTORY_DAYS)
    history_end=start_day-RECENT_DAYS-1
    historical=tx.loc[tx["DAY"].between(history_start,history_end)]
    recent=tx.loc[tx["DAY"].between(start_day-RECENT_DAYS,start_day-1)]
    habit=historical.groupby(["household_key","COMMODITY_DESC"],dropna=False).agg(
        history_baskets=("BASKET_ID","nunique"),history_revenue=("SALES_VALUE","sum"),history_last_day=("DAY","max")
    ).reset_index().query("history_baskets>=@MIN_HISTORY_BASKETS")
    recent_keys=recent[["household_key","COMMODITY_DESC"]].drop_duplicates().assign(recent_purchase=1)
    lapsed=habit.merge(recent_keys,on=["household_key","COMMODITY_DESC"],how="left")
    lapsed=lapsed.loc[lapsed["recent_purchase"].isna()].drop(columns="recent_purchase")
    lapsed["category_recency"]=start_day-lapsed["history_last_day"]
    customer_history=historical.groupby("household_key").agg(customer_history_revenue=("SALES_VALUE","sum"),customer_history_baskets=("BASKET_ID","nunique")).reset_index()
    return lapsed.merge(customer_history,on="household_key",how="left")

def category_outcomes(start_day):
    follow_end=start_day+FOLLOWUP_DAYS-1
    follow=tx.loc[tx["DAY"].between(start_day,follow_end)]
    category=follow.groupby(["household_key","COMMODITY_DESC"],dropna=False).agg(
        category_revenue_56d=("SALES_VALUE","sum"),category_baskets_56d=("BASKET_ID","nunique")
    ).reset_index()
    category["category_repurchase_56d"]=(category["category_baskets_56d"]>0).astype(int)
    overall=follow.groupby("household_key").agg(overall_revenue_56d=("SALES_VALUE","sum"),active_weeks_56d=("WEEK_NO","nunique")).reset_index()
    return category,overall

## 3. 동일한 중단 카테고리 안에서 비교군 매칭

같은 카테고리가 중단된 고객끼리 과거 카테고리 구매횟수·매출·중단 기간과 전체 구매규모가 가장 가까운 비교 고객을 찾습니다.

In [ ]:
MATCH_COLS=["history_baskets","history_revenue","category_recency","customer_history_revenue","customer_history_baskets"]
def match_within_category(treated,controls):
    matched_parts=[]
    for category,t in treated.groupby("COMMODITY_DESC",dropna=False):
        c=controls.loc[controls["COMMODITY_DESC"].eq(category)].reset_index(drop=True)
        if c.empty: continue
        both=pd.concat([t[MATCH_COLS],c[MATCH_COLS]])
        x=np.log1p(both.clip(lower=0).to_numpy(float)); mean,std=x.mean(axis=0),x.std(axis=0); std[std==0]=1
        ta=(np.log1p(t[MATCH_COLS].clip(lower=0).to_numpy(float))-mean)/std
        ca=(np.log1p(c[MATCH_COLS].clip(lower=0).to_numpy(float))-mean)/std
        distance=((ta[:,None,:]-ca[None,:,:])**2).sum(axis=2)
        nearest_index=distance.argmin(axis=1); nearest_distance=distance[np.arange(len(t)),nearest_index]
        keep=nearest_distance<=MATCH_CALIPER_SQUARED
        if not keep.any(): continue
        selected=c.iloc[nearest_index[keep]].reset_index(drop=True).copy()
        selected["treated_row_id"]=t.loc[keep,"treated_row_id"].to_numpy()
        selected["match_distance_squared"]=nearest_distance[keep]
        matched_parts.append(selected)
    return pd.concat(matched_parts,ignore_index=True) if matched_parts else pd.DataFrame()

def smd(a,b):
    pooled=np.sqrt((a.var(ddof=1)+b.var(ddof=1))/2)
    return 0 if pooled==0 else (a.mean()-b.mean())/pooled

## 4. 캠페인별 카테고리 복귀 효과 계산

In [ ]:
campaign_results,pair_results=[],[]
for camp in valid_campaigns.sort_values("CAMPAIGN").itertuples(index=False):
    cid,start=int(camp.CAMPAIGN),int(camp.START_DAY); follow_end=start+FOLLOWUP_DAYS-1
    cohort=lapsed_category_cohort(start); category_out,overall_out=category_outcomes(start)
    cohort=cohort.merge(category_out,on=["household_key","COMMODITY_DESC"],how="left").merge(overall_out,on="household_key",how="left")
    for col in ["category_revenue_56d","category_baskets_56d","category_repurchase_56d","overall_revenue_56d","active_weeks_56d"]: cohort[col]=cohort[col].fillna(0)
    camp_categories=set(coupon_categories.loc[coupon_categories["CAMPAIGN"]==cid,"COMMODITY_DESC"]); assigned=set(campaign_table.loc[campaign_table["CAMPAIGN"]==cid,"household_key"])
    treated=cohort.loc[cohort["household_key"].isin(assigned)&cohort["COMMODITY_DESC"].isin(camp_categories)].reset_index(drop=True)
    overlaps=campaign_desc.loc[(campaign_desc["START_DAY"]<=follow_end)&(campaign_desc["END_DAY"]>=start-RECENT_DAYS),"CAMPAIGN"]
    exposed=set(campaign_table.loc[campaign_table["CAMPAIGN"].isin(overlaps),"household_key"])
    controls=cohort.loc[~cohort["household_key"].isin(exposed)].reset_index(drop=True)
    if len(treated)<MIN_TREATED_PAIRS or len(controls)<MIN_TREATED_PAIRS: continue
    treated=treated.copy(); treated["treated_row_id"]=np.arange(len(treated))
    matched=match_within_category(treated,controls)
    kept=treated.loc[treated["treated_row_id"].isin(matched["treated_row_id"])].sort_values("treated_row_id").reset_index(drop=True)
    matched=matched.sort_values("treated_row_id").reset_index(drop=True)
    if len(kept)<MIN_TREATED_PAIRS: continue
    rep_effect=kept["category_repurchase_56d"].to_numpy()-matched["category_repurchase_56d"].to_numpy()
    cat_rev_effect=kept["category_revenue_56d"].to_numpy()-matched["category_revenue_56d"].to_numpy()
    active_effect=kept["active_weeks_56d"].to_numpy()-matched["active_weeks_56d"].to_numpy()
    overall_rev_effect=kept["overall_revenue_56d"].to_numpy()-matched["overall_revenue_56d"].to_numpy()
    balances={f"smd_{col}":smd(kept[col],matched[col]) for col in MATCH_COLS}
    campaign_results.append({"campaign":cid,"campaign_type":camp.DESCRIPTION,"coupon_categories":camp.coupon_categories,
        "treated_pairs":len(kept),"treated_households":kept["household_key"].nunique(),"unique_control_households":matched["household_key"].nunique(),
        "treated_category_repurchase_rate":kept["category_repurchase_56d"].mean(),"control_category_repurchase_rate":matched["category_repurchase_56d"].mean(),
        "category_repurchase_effect":rep_effect.mean(),"category_revenue_effect":cat_rev_effect.mean(),
        "active_weeks_effect":active_effect.mean(),"overall_revenue_effect":overall_rev_effect.mean(),**balances})
    pair_results.append(pd.DataFrame({"campaign":cid,"household_key":kept["household_key"],"COMMODITY_DESC":kept["COMMODITY_DESC"],
        "control_household":matched["household_key"],"category_repurchase_effect":rep_effect,"category_revenue_effect":cat_rev_effect,
        "active_weeks_effect":active_effect,"overall_revenue_effect":overall_rev_effect}))

lapsed_category_campaign_effects=pd.DataFrame(campaign_results)
lapsed_category_pairs=pd.concat(pair_results,ignore_index=True) if pair_results else pd.DataFrame()
balance_cols=[f"smd_{c}" for c in MATCH_COLS]
lapsed_category_campaign_effects["quality_pass"]=lapsed_category_campaign_effects[balance_cols].abs().max(axis=1)<=.1
display(lapsed_category_campaign_effects)

## 5. 캠페인 단위 가설 검정

In [ ]:
def one_sample_test(values,n_boot=10000,n_perm=20000):
    values=np.asarray(values,float); rng=np.random.default_rng(RANDOM_SEED)
    if len(values)<2:return {"campaigns":len(values),"mean_effect":np.nan,"ci_low":np.nan,"ci_high":np.nan,"p_value_one_sided":np.nan}
    boot=rng.choice(values,size=(n_boot,len(values)),replace=True).mean(axis=1)
    signs=rng.choice([-1,1],size=(n_perm,len(values))); null=(signs*values).mean(axis=1); obs=values.mean()
    return {"campaigns":len(values),"mean_effect":obs,"ci_low":np.quantile(boot,.025),"ci_high":np.quantile(boot,.975),
            "p_value_one_sided":(1+(null>=obs).sum())/(n_perm+1)}
qualified=lapsed_category_campaign_effects.loc[lapsed_category_campaign_effects["quality_pass"]].copy()
tests=[]
for outcome,col in [("카테고리 재구매율","category_repurchase_effect"),("카테고리 56일 매출","category_revenue_effect"),
                    ("전체 8주 활동주차","active_weeks_effect"),("전체 56일 매출","overall_revenue_effect")]:
    tests.append({"outcome":outcome,**one_sample_test(qualified[col])})
lapsed_category_hypothesis_tests=pd.DataFrame(tests)
display(lapsed_category_hypothesis_tests)
primary=lapsed_category_hypothesis_tests.iloc[0]
supported=primary["mean_effect"]>0 and primary["p_value_one_sided"]<.05
print("판정: 가설을 지지합니다." if supported else "판정: 현재 데이터로 가설을 지지할 충분한 근거가 없습니다.")

## 6. 해석 주의사항

- 개별 가구가 캠페인 내 특정 쿠폰을 실제로 받았는지는 없으므로 캠페인 쿠폰 목록에 카테고리가 포함된 것을 노출로 간주합니다.
- 캠페인 배정은 무작위가 아니므로 매칭 후에도 관측되지 않은 차이가 남을 수 있습니다.
- 한 고객이 여러 중단 카테고리를 가질 수 있어 고객×카테고리 쌍을 분석하되, 검정은 캠페인별 평균 효과를 단위로 수행합니다.
- 가설이 지지돼도 실제 도입 전 무작위 A/B 테스트가 필요합니다.

In [ ]:
lapsed_category_campaign_effects.to_csv(OUTPUT_DIR/"lapsed_category_campaign_effects.csv",index=False,encoding="utf-8-sig")
lapsed_category_hypothesis_tests.to_csv(OUTPUT_DIR/"lapsed_category_hypothesis_tests.csv",index=False,encoding="utf-8-sig")
lapsed_category_pairs.to_csv(OUTPUT_DIR/"lapsed_category_customer_pairs.csv",index=False,encoding="utf-8-sig")
print("가설 검증 결과 3개 저장 완료")